# PM2.5-GNN — all-models comparison, one config

Runs every model in `MODELS` on **one** fixed `(dataset_num, hist_len, pred_len)` config. Edit `DATASET_NUM`, `HIST_LEN`, `PRED_LEN` in the **CONFIG cell below** (that's the only thing you should need to touch) and run all cells once — every model trains automatically, back to back, no further clicks.

For each model it trains `EXP_REPEAT` times (matching `train.py`'s `exp_repeat`), averages the metrics, and writes:

- `output/{MODEL}_ds{N}_hl{H}_pl{P}.txt` — a detailed report per model (same format as the main project's `metric.txt`)
- `output/summary.csv` — one row per model, all models side by side, for quick comparison

**Setup is identical to `train.ipynb`:** upload this whole `notebooks/` folder so `config.yaml`, `util.py`, `graph.py`, `dataset.py`, `model/`, `layers/`, `data/`, and `KnowAir.npy` sit next to this notebook. See `notebooks/README.md` for the full file list.

**Before you run this, know what you're starting:** `len(MODELS) * EXP_REPEAT` full training runs, each up to `EPOCHS` epochs — with the defaults (16 models × 5 repeats = 80 runs) that's a long CPU run, and a few models here (`AirDDE`, `AirPhyNet`, `AirDualODE`) use an ODE solver and are noticeably slower per batch than the rest. Trim `MODELS`/`EPOCHS`/`EXP_REPEAT` for a first test pass, or drop the slow ones from `MODELS` if you just want a quick comparison.

**If the config itself doesn't have enough data:** `dataset_num` 2 and 3 cover only a few months each. With a large `pred_len` relative to `HIST_LEN`, a split's val/test window can end up too short to window at all, or too short to fill even one `batch_size`-sized batch. Since `(dataset_num, hist_len, pred_len)` is fixed for the whole run here, that's checked once up front (not per model) — if it fails, the notebook stops with a clear message before training anything, instead of failing partway through.

In [ ]:
# Install dependencies not already present on this JupyterLab instance.
# Safe to re-run; comment out if your environment already has everything.
%pip install -q -r requirements.txt

In [ ]:
import os
import sys

notebook_dir = os.getcwd()
if notebook_dir not in sys.path:
    sys.path.append(notebook_dir)

import csv
import time
import traceback

import arrow
import numpy as np
import torch
from torch import nn
from tqdm.auto import tqdm

from util import config, file_dir
from graph import Graph
from dataset import HazeData

from model.MLP import MLP
from model.LSTM import LSTM
from model.GRU import GRU
from model.GC_LSTM import GC_LSTM
from model.nodesFC_GRU import nodesFC_GRU
from model.PM25_GNN import PM25_GNN
from model.PM25_GNN_nosub import PM25_GNN_nosub
from model.airformer import AirFormerPM25
from model.informer import InformerPM25
from model.autoformer import AutoformerPM25
from model.patchtst import PatchTSTPM25
from model.staeformer import STAEformerPM25
from model.airdde import AirDDEPM25
from model.airphynet import AirPhyNetPM25
from model.airdualode import AirDualODEPM25
from model.probgru8 import ProbGRUModel8

torch.set_num_threads(1)
use_cuda = torch.cuda.is_available()
device = torch.device('cuda' if use_cuda else 'cpu')
print('device:', device)

## CONFIG — the cell you need to edit

Change `DATASET_NUM`, `HIST_LEN`, `PRED_LEN` for the run you want. `MODELS` defaults to every model available in `notebooks/model/`; trim it if you only want a subset.

In [ ]:
DATASET_NUM = 3     # config.yaml's dataset: key (1, 2, or 3)
HIST_LEN = 24        # lookback window
PRED_LEN = 8         # forecast horizon

MODELS = [
    'MLP', 'LSTM', 'GRU', 'GC_LSTM', 'nodesFC_GRU',
    'PM25_GNN', 'PM25_GNN_nosub', 'AirFormer', 'Informer', 'Autoformer',
    'PatchTST', 'STAEformer', 'AirDDE', 'AirPhyNet', 'AirDualODE', 'ProbGRUModel8',
]

EXP_REPEAT = config['train']['exp_repeat']   # repeats per model, averaged (config.yaml default: 5)
EPOCHS = config['train']['epochs']           # max epochs per repeat (config.yaml default: 50)
BATCH_SIZE = config['train']['batch_size']
EARLY_STOP = config['train']['early_stop']
LR = config['train']['lr']
WEIGHT_DECAY = config['train']['weight_decay']

SAVE_NPY = False           # save predict/label/time .npy per repeat (can add up fast across many models)
SAVE_CHECKPOINTS = False   # save model.pth per repeat

OUTPUT_DIR = os.path.join(notebook_dir, 'output')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'{len(MODELS)} model(s) x {EXP_REPEAT} repeat(s) = {len(MODELS) * EXP_REPEAT} total training run(s)')
print('output dir:', OUTPUT_DIR)

In [ ]:
graph = Graph()
city_num = graph.node_num
results_dir = file_dir['results_dir']  # only used if SAVE_CHECKPOINTS is True
criterion = nn.MSELoss()
kl_weight = config['train'].get('kl_weight', 0.01)
alignment_weight = config['train'].get('alignment_weight', 0.1)

idx, citys, lons, lats = graph.traverse_graph()
coords = torch.tensor(np.stack([lats, lons], axis=1), dtype=torch.float32)
altitude = torch.tensor(graph.node_attr[:, 0], dtype=torch.float32)

print('city_num:', city_num, '| device:', device)

In [ ]:
# hist_len / pred_len / dataset_num / batch_size are fixed for the whole run (set from the
# CONFIG cell). train_data/val_data/test_data are built ONCE here and reused for every model -
# only exp_model changes per model, set as a global by run_for_model() below.
hist_len, pred_len, dataset_num, batch_size = HIST_LEN, PRED_LEN, DATASET_NUM, BATCH_SIZE

try:
    # HazeData._add_time_dim asserts each split's raw length > hist_len+pred_len - a large
    # pred_len against a short dataset_num (2, 3 cover a few months each) can violate that.
    train_data = HazeData(graph, hist_len, pred_len, dataset_num, flag='Train')
    val_data = HazeData(graph, hist_len, pred_len, dataset_num, flag='Val')
    test_data = HazeData(graph, hist_len, pred_len, dataset_num, flag='Test')
except AssertionError as e:
    raise RuntimeError(
        f'dataset_num={dataset_num} is too short to window at hist_len={hist_len}+pred_len={pred_len}. '
        f'Lower PRED_LEN/HIST_LEN or pick a different DATASET_NUM in the CONFIG cell above, then re-run from there.'
    ) from e

counts = {
    'train': len(train_data) // BATCH_SIZE,
    'val': len(val_data) // BATCH_SIZE,
    'test': len(test_data) // BATCH_SIZE,
}
print('batches/epoch:', counts, '(need >=1 in each)')
if min(counts.values()) < 1:
    raise RuntimeError(
        f'Not enough data for batch_size={BATCH_SIZE} at dataset_num={dataset_num}, '
        f'hist_len={hist_len}, pred_len={pred_len} (batches/epoch: {counts}). '
        f'Lower BATCH_SIZE/PRED_LEN/HIST_LEN or pick a different DATASET_NUM in the CONFIG cell above, then re-run from there.'
    )

in_dim = train_data.feature.shape[-1] + train_data.pm25.shape[-1]
wind_mean, wind_std = train_data.wind_mean, train_data.wind_std
pm25_mean, pm25_std = test_data.pm25_mean, test_data.pm25_std

print('in_dim:', in_dim, '| train/val/test sizes:', len(train_data), len(val_data), len(test_data))
print(f'Train: {train_data.start_time} --> {train_data.end_time}')
print(f'Val:   {val_data.start_time} --> {val_data.end_time}')
print(f'Test:  {test_data.start_time} --> {test_data.end_time}')

In [ ]:
# get_model() reads exp_model (reassigned per model by run_for_model() below) plus the
# hist_len/pred_len/in_dim/city_num/batch_size/wind_mean/wind_std/coords/altitude/train_data
# set once above. This is train.ipynb's get_model() unchanged.
def get_model():
    if exp_model == 'MLP':
        return MLP(hist_len, pred_len, in_dim)
    elif exp_model == 'LSTM':
        return LSTM(hist_len, pred_len, in_dim, city_num, batch_size, device)
    elif exp_model == 'GRU':
        return GRU(hist_len, pred_len, in_dim, city_num, batch_size, device)
    elif exp_model == 'nodesFC_GRU':
        return nodesFC_GRU(hist_len, pred_len, in_dim, city_num, batch_size, device)
    elif exp_model == 'GC_LSTM':
        return GC_LSTM(hist_len, pred_len, in_dim, city_num, batch_size, device, graph.edge_index)
    elif exp_model == 'PM25_GNN':
        return PM25_GNN(hist_len, pred_len, in_dim, city_num, batch_size, device, graph.edge_index, graph.edge_attr, wind_mean, wind_std)
    elif exp_model == 'PM25_GNN_nosub':
        return PM25_GNN_nosub(hist_len, pred_len, in_dim, city_num, batch_size, device, graph.edge_index, graph.edge_attr, wind_mean, wind_std)
    elif exp_model == 'AirFormer':
        return AirFormerPM25(
            hist_len, pred_len, in_dim, city_num, batch_size, device,
            graph.edge_index, graph.edge_attr, wind_mean, wind_std,
            station_coords=coords,
            hidden_channels=config['experiments'].get('airformer_hidden_channels', 32),
            end_channels=config['experiments'].get('airformer_end_channels', 512),
            blocks=config['experiments'].get('airformer_blocks', 4),
            num_heads=config['experiments'].get('airformer_num_heads', 2),
            mlp_expansion=config['experiments'].get('airformer_mlp_expansion', 2),
            depth=config['experiments'].get('airformer_depth', 1),
            dropout=config['experiments'].get('airformer_dropout', 0.3),
            spatial_flag=config['experiments'].get('airformer_spatial_flag', True),
            stochastic_flag=config['experiments'].get('airformer_stochastic_flag', True),
        )
    elif exp_model == 'Informer':
        return InformerPM25(
            hist_len, pred_len, in_dim, city_num, batch_size, device,
            label_len=config['experiments'].get('informer_label_len', None),
            d_model=config['experiments'].get('informer_d_model', 64),
            n_heads=config['experiments'].get('informer_n_heads', 8),
            e_layers=config['experiments'].get('informer_e_layers', 2),
            d_layers=config['experiments'].get('informer_d_layers', 1),
            d_ff=config['experiments'].get('informer_d_ff', 256),
            factor=config['experiments'].get('informer_factor', 5),
            dropout=config['experiments'].get('informer_dropout', 0.1),
            attn=config['experiments'].get('informer_attn', 'prob'),
            activation=config['experiments'].get('informer_activation', 'gelu'),
            distil=config['experiments'].get('informer_distil', True),
            mix=config['experiments'].get('informer_mix', True),
        )
    elif exp_model == 'Autoformer':
        return AutoformerPM25(
            hist_len, pred_len, in_dim, city_num, batch_size, device,
            label_len=config['experiments'].get('autoformer_label_len', None),
            d_model=config['experiments'].get('autoformer_d_model', 64),
            n_heads=config['experiments'].get('autoformer_n_heads', 8),
            e_layers=config['experiments'].get('autoformer_e_layers', 2),
            d_layers=config['experiments'].get('autoformer_d_layers', 1),
            d_ff=config['experiments'].get('autoformer_d_ff', 256),
            moving_avg=config['experiments'].get('autoformer_moving_avg', 25),
            factor=config['experiments'].get('autoformer_factor', 1),
            dropout=config['experiments'].get('autoformer_dropout', 0.1),
            activation=config['experiments'].get('autoformer_activation', 'gelu'),
        )
    elif exp_model == 'PatchTST':
        return PatchTSTPM25(
            hist_len, pred_len, in_dim, city_num, batch_size, device,
            patch_len=config['experiments'].get('patchtst_patch_len', 8),
            stride=config['experiments'].get('patchtst_stride', 4),
            d_model=config['experiments'].get('patchtst_d_model', 32),
            n_heads=config['experiments'].get('patchtst_n_heads', 4),
            e_layers=config['experiments'].get('patchtst_e_layers', 2),
            d_ff=config['experiments'].get('patchtst_d_ff', 128),
            dropout=config['experiments'].get('patchtst_dropout', 0.1),
            head_dropout=config['experiments'].get('patchtst_head_dropout', 0.1),
            activation=config['experiments'].get('patchtst_activation', 'gelu'),
            revin=config['experiments'].get('patchtst_revin', True),
        )
    elif exp_model == 'STAEformer':
        return STAEformerPM25(
            hist_len, pred_len, in_dim, city_num, batch_size, device,
            feature_mean=train_data.feature_mean,
            feature_std=train_data.feature_std,
            input_embedding_dim=config['experiments'].get('staeformer_input_dim', 24),
            tod_embedding_dim=config['experiments'].get('staeformer_tod_dim', 24),
            dow_embedding_dim=config['experiments'].get('staeformer_dow_dim', 24),
            adaptive_embedding_dim=config['experiments'].get('staeformer_adaptive_dim', 80),
            n_heads=config['experiments'].get('staeformer_n_heads', 4),
            e_layers=config['experiments'].get('staeformer_e_layers', 3),
            d_ff=config['experiments'].get('staeformer_d_ff', 256),
            dropout=config['experiments'].get('staeformer_dropout', 0.1),
            activation=config['experiments'].get('staeformer_activation', 'gelu'),
            dt_hours=config['experiments'].get('staeformer_dt_hours', 3.0),
        )
    elif exp_model == 'AirDDE':
        return AirDDEPM25(
            hist_len, pred_len, in_dim, city_num, batch_size, device,
            graph.edge_index, graph.edge_attr, wind_mean, wind_std,
            rnn_units=config['experiments'].get('airdde_rnn_units', 64),
            rnn_num_layers=config['experiments'].get('airdde_rnn_num_layers', 1),
            agcn_cheb_k=config['experiments'].get('airdde_agcn_cheb_k', 3),
            mem_num=config['experiments'].get('airdde_mem_num', 20),
            mem_dim=config['experiments'].get('airdde_mem_dim', 64),
            local_mem_tau=config['experiments'].get('airdde_local_mem_tau', 3),
            local_mem_k=config['experiments'].get('airdde_local_mem_k', 8),
            ode_gcn_hidden_dim=config['experiments'].get('airdde_ode_gcn_hidden_dim', 64),
            ode_cheb_k=config['experiments'].get('airdde_ode_cheb_k', 3),
            ode_num_layers=config['experiments'].get('airdde_ode_num_layers', 3),
            ode_method=config['experiments'].get('airdde_ode_method', 'dopri5'),
            ode_rtol=config['experiments'].get('airdde_ode_rtol', 1e-2),
            ode_atol=config['experiments'].get('airdde_ode_atol', 1e-2),
            ode_adjoint=config['experiments'].get('airdde_ode_adjoint', True),
        )
    elif exp_model == 'AirPhyNet':
        return AirPhyNetPM25(
            hist_len, pred_len, in_dim, city_num, batch_size, device,
            graph.edge_index, graph.edge_attr, wind_mean, wind_std,
            rnn_units=config['experiments'].get('airphynet_rnn_units', 64),
            latent_dim=config['experiments'].get('airphynet_latent_dim', 4),
            gcn_step=config['experiments'].get('airphynet_gcn_step', 2),
            diff_coeff=config['experiments'].get('airphynet_diff_coeff', 0.1),
            n_traj_samples=config['experiments'].get('airphynet_n_traj_samples', 1),
            ode_method=config['experiments'].get('airphynet_ode_method', 'dopri5'),
            ode_rtol=config['experiments'].get('airphynet_ode_rtol', 1e-3),
            ode_atol=config['experiments'].get('airphynet_ode_atol', 1e-4),
            ode_adjoint=config['experiments'].get('airphynet_ode_adjoint', True),
            filter_type=config['experiments'].get('airphynet_filter_type', 'diff_adv'),
        )
    elif exp_model == 'AirDualODE':
        return AirDualODEPM25(
            hist_len, pred_len, in_dim, city_num, batch_size, device,
            graph.edge_index, graph.edge_attr, wind_mean, wind_std,
            phy_latent_dim=config['experiments'].get('airdualode_phy_latent_dim', 8),
            unk_latent_dim=config['experiments'].get('airdualode_unk_latent_dim', 8),
            fusion_dim=config['experiments'].get('airdualode_fusion_dim', 16),
            gcn_step=config['experiments'].get('airdualode_gcn_step', 2),
            rnn_units=config['experiments'].get('airdualode_rnn_units', 32),
            attn_heads=config['experiments'].get('airdualode_attn_heads', 2),
            estimate_coeff=config['experiments'].get('airdualode_estimate_coeff', False),
            ode_method=config['experiments'].get('airdualode_ode_method', 'dopri5'),
            ode_rtol=config['experiments'].get('airdualode_ode_rtol', 1e-3),
            ode_atol=config['experiments'].get('airdualode_ode_atol', 1e-4),
            ode_adjoint=config['experiments'].get('airdualode_ode_adjoint', True),
        )
    elif exp_model == 'ProbGRUModel8':
        return ProbGRUModel8(
            hist_len, pred_len, in_dim, city_num, batch_size, device,
            graph.edge_index, graph.edge_attr, wind_mean, wind_std,
            station_coords=coords,
            station_elevation=altitude,
            feature_mean=train_data.feature_mean,
            feature_std=train_data.feature_std,
            hidden_dim=config['experiments'].get('gru_hidden_dim', 64),
            latent_dim=config['experiments'].get('gru_latent_dim', 16),
            attn_dim=config['experiments'].get('gru_attn_dim', 32),
            num_layers=config['experiments'].get('gru_num_layers', 1),
            dropout=config['experiments'].get('gru_dropout', 0.1),
            logvar_clamp=config['experiments'].get('gru_logvar_clamp', 10.0),
            spatial_mix_mode=config['experiments'].get('gru_spatial_mix_mode', 'bottleneck'),
            max_lag=config['experiments'].get('gru_max_lag', 6),
            dist_threshold_km=config['experiments'].get('gru_dist_threshold_km', 300.0),
            sigma_d=config['experiments'].get('gru_sigma_d', 200.0),
            sigma_h=config['experiments'].get('gru_sigma_h', 1200.0),
            sigma_tau_init_h=config['experiments'].get('gru_sigma_tau_init_h', 3.0),
            dt_hours=config['experiments'].get('gru_dt_hours', 3.0),
        )
    else:
        raise Exception('Wrong model name!')

In [ ]:
# get_metric / train / val / test / get_mean_std - unchanged from train.ipynb.
def get_metric(predict_epoch, label_epoch):
    haze_threshold = 75
    predict_haze = predict_epoch >= haze_threshold
    predict_clear = predict_epoch < haze_threshold
    label_haze = label_epoch >= haze_threshold
    label_clear = label_epoch < haze_threshold
    hit = np.sum(np.logical_and(predict_haze, label_haze))
    miss = np.sum(np.logical_and(label_haze, predict_clear))
    falsealarm = np.sum(np.logical_and(predict_haze, label_clear))
    csi = hit / (hit + falsealarm + miss)
    pod = hit / (hit + miss)
    far = falsealarm / (hit + falsealarm)
    predict = predict_epoch[:,:,:,0].transpose((0,2,1))
    label = label_epoch[:,:,:,0].transpose((0,2,1))
    predict = predict.reshape((-1, predict.shape[-1]))
    label = label.reshape((-1, label.shape[-1]))
    mae = np.mean(np.mean(np.abs(predict - label), axis=1))
    rmse = np.mean(np.sqrt(np.mean(np.square(predict - label), axis=1)))
    return rmse, mae, csi, pod, far


def train(train_loader, model, optimizer):
    model.train()
    train_loss = 0
    for batch_idx, data in tqdm(enumerate(train_loader), total=len(train_loader), leave=False):
        optimizer.zero_grad()
        pm25, feature, time_arr = data
        pm25 = pm25.to(device)
        feature = feature.to(device)
        pm25_label = pm25[:, hist_len:]
        pm25_hist = pm25[:, :hist_len]
        pm25_pred = model(pm25_hist, feature)
        loss = criterion(pm25_pred, pm25_label)
        kl = getattr(model, 'last_kl_loss', None)
        if kl is not None:
            loss = loss + kl_weight * kl
        alignment = getattr(model, 'last_alignment_loss', None)
        if alignment is not None:
            loss = loss + alignment_weight * alignment
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= batch_idx + 1
    return train_loss


def val(val_loader, model):
    model.eval()
    val_loss = 0
    for batch_idx, data in tqdm(enumerate(val_loader), total=len(val_loader), leave=False):
        pm25, feature, time_arr = data
        pm25 = pm25.to(device)
        feature = feature.to(device)
        pm25_label = pm25[:, hist_len:]
        pm25_hist = pm25[:, :hist_len]
        pm25_pred = model(pm25_hist, feature)
        loss = criterion(pm25_pred, pm25_label)
        val_loss += loss.item()
    val_loss /= batch_idx + 1
    return val_loss


def test(test_loader, model):
    model.eval()
    predict_list = []
    label_list = []
    time_list = []
    test_loss = 0
    for batch_idx, data in enumerate(test_loader):
        pm25, feature, time_arr = data
        pm25 = pm25.to(device)
        feature = feature.to(device)
        pm25_label = pm25[:, hist_len:]
        pm25_hist = pm25[:, :hist_len]
        pm25_pred = model(pm25_hist, feature)
        loss = criterion(pm25_pred, pm25_label)
        test_loss += loss.item()

        pm25_pred_val = np.concatenate([pm25_hist.cpu().detach().numpy(), pm25_pred.cpu().detach().numpy()], axis=1) * pm25_std + pm25_mean
        pm25_label_val = pm25.cpu().detach().numpy() * pm25_std + pm25_mean
        predict_list.append(pm25_pred_val)
        label_list.append(pm25_label_val)
        time_list.append(time_arr.cpu().detach().numpy())

    test_loss /= batch_idx + 1
    predict_epoch = np.concatenate(predict_list, axis=0)
    label_epoch = np.concatenate(label_list, axis=0)
    time_epoch = np.concatenate(time_list, axis=0)
    predict_epoch[predict_epoch < 0] = 0
    return test_loss, predict_epoch, label_epoch, time_epoch


def get_mean_std(data_list):
    data = np.asarray(data_list)
    return data.mean(), data.std()

In [ ]:
def run_single_repeat(model, optimizer, train_loader, val_loader, test_loader, repeat_dir):
    """One exp_repeat iteration: train up to EPOCHS with early stopping on val_loss,
    evaluate on test whenever val improves. Same logic as train.ipynb's main() inner loop.
    Returns a dict of the metrics at the best epoch."""
    val_loss_min = float('inf')
    best_epoch = 0
    best = None

    for epoch in range(EPOCHS):
        train_loss = train(train_loader, model, optimizer)
        val_loss = val(val_loader, model)

        if epoch - best_epoch > EARLY_STOP:
            break

        if val_loss < val_loss_min:
            val_loss_min = val_loss
            best_epoch = epoch
            test_loss, predict_epoch, label_epoch, time_epoch = test(test_loader, model)
            rmse, mae, csi, pod, far = get_metric(predict_epoch, label_epoch)
            best = dict(train_loss=train_loss, val_loss=val_loss, test_loss=test_loss,
                        rmse=rmse, mae=mae, csi=csi, pod=pod, far=far, epoch=epoch)

            if SAVE_CHECKPOINTS or SAVE_NPY:
                os.makedirs(repeat_dir, exist_ok=True)
            if SAVE_CHECKPOINTS:
                torch.save(model.state_dict(), os.path.join(repeat_dir, 'model.pth'))
            if SAVE_NPY:
                np.save(os.path.join(repeat_dir, 'predict.npy'), predict_epoch)
                np.save(os.path.join(repeat_dir, 'label.npy'), label_epoch)
                np.save(os.path.join(repeat_dir, 'time.npy'), time_epoch)

    if best is None:
        # val_loss never improved even once (e.g. EARLY_STOP triggered on epoch 0) -
        # fall back to the last epoch's numbers so this model still reports something.
        test_loss, predict_epoch, label_epoch, time_epoch = test(test_loader, model)
        rmse, mae, csi, pod, far = get_metric(predict_epoch, label_epoch)
        best = dict(train_loss=train_loss, val_loss=val_loss, test_loss=test_loss,
                    rmse=rmse, mae=mae, csi=csi, pod=pod, far=far, epoch=epoch)
    return best

In [ ]:
METRIC_KEYS = ['train_loss', 'val_loss', 'test_loss', 'rmse', 'mae', 'csi', 'pod', 'far']


def run_for_model(model_name):
    """Trains model_name on the shared (dataset_num, hist_len, pred_len) EXP_REPEAT times and
    writes output/{model_name}_ds{N}_hl{H}_pl{P}.txt. Returns a summary dict (also used for
    output/summary.csv)."""
    global exp_model
    exp_model = model_name
    tag = f'{model_name}_ds{dataset_num}_hl{hist_len}_pl{pred_len}'
    print(f'\n=== {tag} ===')

    combo_dir = os.path.join(OUTPUT_DIR, 'artifacts', tag)
    per_repeat = {k: [] for k in METRIC_KEYS}
    n_ok = 0
    model_repr = ''

    for exp_idx in range(EXP_REPEAT):
        print(f'  repeat {exp_idx}...', end=' ', flush=True)
        t0 = time.time()
        train_loader = torch.utils.data.DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
        val_loader = torch.utils.data.DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False, drop_last=True)
        test_loader = torch.utils.data.DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False, drop_last=True)
        try:
            model = get_model().to(device)
            model_repr = str(model)
            optimizer = torch.optim.RMSprop(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
            repeat_dir = os.path.join(combo_dir, f'rep{exp_idx:02d}')
            result = run_single_repeat(model, optimizer, train_loader, val_loader, test_loader, repeat_dir)
            for k in METRIC_KEYS:
                per_repeat[k].append(result[k])
            n_ok += 1
            print(f'test RMSE={result["rmse"]:.3f} MAE={result["mae"]:.3f} ({time.time()-t0:.0f}s)')
        except Exception as e:
            print(f'FAILED: {type(e).__name__}: {e}')
            traceback.print_exc(limit=3)

    summary = dict(model=model_name, dataset_num=dataset_num, hist_len=hist_len, pred_len=pred_len,
                   status='ok' if n_ok else 'all_repeats_failed', n_repeats_ok=n_ok)
    for k in METRIC_KEYS:
        mean, std = (get_mean_std(per_repeat[k]) if per_repeat[k] else (float('nan'), float('nan')))
        summary[f'{k}_mean'] = mean
        summary[f'{k}_std'] = std

    report = (
        f'============== Model comparison report ==============\n'
        f'Model: {model_name}\n'
        f'dataset_num: {dataset_num}  hist_len: {hist_len}  pred_len: {pred_len}\n'
        f'Train: {train_data.start_time} --> {train_data.end_time}\n'
        f'Val:   {val_data.start_time} --> {val_data.end_time}\n'
        f'Test:  {test_data.start_time} --> {test_data.end_time}\n'
        f'City number: {city_num}\n'
        f'batch_size: {BATCH_SIZE}  epochs(max): {EPOCHS}  early_stop: {EARLY_STOP}  lr: {LR}\n'
        f'exp_repeat requested: {EXP_REPEAT}  succeeded: {n_ok}\n'
        f'=======================================================\n'
        + '\n'.join(f'{k:10s} | mean: {summary[f"{k}_mean"]:.4f}  std: {summary[f"{k}_std"]:.4f}' for k in METRIC_KEYS) + '\n'
        f'=======================================================\n{model_repr}\n'
    )
    print(report)
    with open(os.path.join(OUTPUT_DIR, f'{tag}.txt'), 'w') as f:
        f.write(report)

    return summary

In [ ]:
summary_fp = os.path.join(OUTPUT_DIR, 'summary.csv')
fieldnames = ['model', 'dataset_num', 'hist_len', 'pred_len', 'status', 'n_repeats_ok']
for k in METRIC_KEYS:
    fieldnames += [f'{k}_mean', f'{k}_std']

all_summaries = []
with open(summary_fp, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for model_name in MODELS:
        try:
            summary = run_for_model(model_name)
        except Exception as e:
            print(f'{model_name} FAILED entirely: {type(e).__name__}: {e}')
            traceback.print_exc(limit=3)
            summary = dict(model=model_name, dataset_num=dataset_num, hist_len=hist_len, pred_len=pred_len,
                           status='error', n_repeats_ok=0)
        all_summaries.append(summary)
        writer.writerow({k: summary.get(k, '') for k in fieldnames})
        f.flush()  # persist progress after every model, not just at the end

failed = [s for s in all_summaries if s['status'] != 'ok']
if failed:
    with open(os.path.join(OUTPUT_DIR, 'failed.txt'), 'w') as f:
        for s in failed:
            f.write(f"{s['model']}: {s['status']}\n")

print(f'\nDone. {len(all_summaries) - len(failed)}/{len(all_summaries)} models completed.')
print('summary:', summary_fp)
if failed:
    print('failed models logged in output/failed.txt:', [(s['model'], s['status']) for s in failed])